

# 전이학습과 딥러닝 경량화

## 딥러닝 모델의 한계

### ① 데이터가 부족해요

- 예를 들어 병원에서 흉부 X-ray를 보고 정상과 폐렴을 분류하는 AI 모델을 만들고 싶다고 가정한다.
- 하지만 흉부 X-ray 이미지가 100장뿐이라면 다양한 질환 패턴을 배우기 어렵다.

### ② 라벨링이 어려워요

- 데이터가 부족한 것과 더불어 정답 라벨을 만드는 과정도 어렵다.
- 흉부 X-ray의 정상과 폐렴 여부를 구분하려면 의료 전문가의 판단이 필요하다.
- 학습에 사용할 충분한 라벨 데이터를 확보하기 어렵다.

### ③ 모델 학습 시 많은 시간과 비용이 소모돼요

- 새로운 문제를 해결하기 위해 모델을 처음부터 다시 학습해야 한다면 학습에 많은 시간과 비용이 소모된다.

> 이미 수백만 장의 이미지를 학습한 AI의 이미지 특징 파악 능력을 가져와서 흉부 X-ray로 정상·폐렴 구분에 활용할 수는 없을까?

## 전이학습의 정의

### ① 전이학습이란?

**전이학습(Transfer Learning)**은 모델의 사전 학습을 통해 배운 능력을 활용해 새로운 작업을 학습하는 것이다.

예를 들어 고양이와 강아지 사진으로 사전 학습한 모델을 흉부 X-ray 분류에 활용할 수 있다.

1. **동물 사진으로 모델 사전 학습하기**
   - 고양이와 강아지 사진으로 사전 학습된 모델은 두 동물을 구분하기 위해 윤곽, 질감, 밝기와 같은 특징을 파악하는 법을 학습한다.
   - 이때 특징을 찾는 방법은 모델의 가중치에 담기게 된다.
2. **정상·폐렴 구분에 흉부 X-ray 적용하기**
   - 기존에 학습된 모델에 흉부 X-ray 데이터를 적용한다.
   - 기존 모델이 학습한 이미지 특징 추출 능력을 활용한다.
3. **정답이 있는 X-ray로 학습하기**
   - 정상과 폐렴을 구분하는 새로운 분류층을 학습한다.
   - 기존 모델이 파악한 특징 중 어떤 조합이 정상과 폐렴을 구분하는 데 중요한지 학습한다.
4. **Fine-tuning(필요시)**
   - 필요한 경우 모델의 가중치도 X-ray에 맞게 조정할 수 있다.

> 사전 학습된 모델을 사용하면 모델 전체를 처음부터 학습시킬 필요 없이 새 작업을 진행할 수 있다.

### ② 전이학습의 4가지 핵심 용어

| 용어 | 뜻 | 예시 |
|---|---|---|
| **Source** | 기존에 학습한 문제 | 고양이와 강아지 분류 문제 |
| **Target** | 새롭게 해결할 문제 | X-ray 정상·폐렴 분류 문제 |
| **Domain** | 사용할 데이터의 특성 | 일반 동물 사진 / X-ray |
| **Task** | 수행하려는 작업 | 고양이·강아지 분류 / 정상·폐렴 분류 |

- Source와 Target의 Domain이나 Task 중 적어도 하나가 다를 때, Source에서 배운 지식을 Target에 활용하는 것이 전이학습이다.
- Source와 Target의 Domain을 각각 `D_S`, `D_T`, Task를 각각 `T_S`, `T_T`로 표현한다.

## 1-3. 전이학습의 분류

### ① Domain과 Task에 따른 분류

| 종류 | Domain | Task | 정답 레이블 | 예시 |
|---|---|---|---|---|
| 귀납적(Inductive) | Source와 Target이 같거나 다를 수 있음 | Source와 Target이 다름 | Source: O 또는 X / Target: O | 이미지 분류 → 객체 위치 탐지 |
| 전이적(Transductive) | Source와 Target이 다름 | Source와 Target이 같음 | Source: O / Target: X | 뉴스 감정 분류 → 다른 언어권 뉴스 감정 분류 |
| 비지도(Unsupervised) | Source와 Target이 다름 | Source와 Target 모두 비지도 작업 | Source: X / Target: X | 한 분야의 군집화 지식 → 다른 분야의 군집화 활용 |

#### 귀납적 전이학습

- 기존에 학습했던 모델의 능력을 바탕으로 정답 레이블이 존재하는 사례를 학습하면서, 새로운 이미지도 분류할 수 있도록 하는 방식이다.
- Source와 Target의 Domain은 같거나 다를 수 있지만, Task는 달라야 한다.
- Target의 정답 레이블은 필요하며, Source의 정답 레이블은 있어도 되고 없어도 된다.

**Source와 Target의 Domain은 같고 Task가 다른 경우**

- Source와 Target 모두 고양이 사진을 사용한다.
- Source의 Task는 사진 속 고양이의 존재 여부를 파악하는 것이고, Target의 Task는 고양이의 위치를 찾는 것이다.
- Domain은 같지만 Task가 다르므로 Target의 정답 레이블이 필요하다.

**Source와 Target의 Domain과 Task가 모두 다른 경우**

- Source는 일반 동물 사진을 사용해 고양이와 강아지를 분류한다.
- Target은 흉부 X-ray를 사용해 정상과 폐렴을 분류한다.
- Domain과 Task가 모두 다르므로 Target의 정답 레이블이 필요하다.

### ② 전이 대상에 따른 분류

전이 대상은 새 작업을 배울 때 기존 모델에서 가져오는 대상이다.

| 방법 | 전이 대상 |
|---|---|
| 인스턴스 기반 | 기존 데이터의 일부 |
| 특징 기반 | 데이터의 특징 표현 |
| 파라미터 기반 | 기존 모델의 학습된 가중치·파라미터 |
| 관계 기반 | 데이터 간 객체 사이의 관계 패턴 |

## 전이학습의 현재 패러다임: Pre-training & Fine-tuning

### ① 전통적인 전이학습에서 가중치와 언어 백터 재사용

- 이미지 분야에서는 ImageNet처럼 대규모 이미지 데이터로 CNN을 사전학습한다.
- CNN은 사물의 특징을 파악하고, 형태·질감·모양과 같은 특징을 뽑아내는 법을 학습한다.
- 새로운 이미지 분류 작업에서는 이 가중치를 가져와 사용하고, 마지막 분류층은 새 작업에 맞게 교체한다.
- 예를 들어 정상·폐렴 라벨이 붙은 흉부 X-ray로 분류층을 학습하고, 필요하다면 기존 모델의 가중치도 Fine-tuning한다.

### ② 트랜스포머 이후의 전이학습: 사전 학습 모델을 여러 작업에 활용

**이미지 분야**

- 대규모 이미지로 사전 학습한 모델을 여러 작업의 출발점으로 사용한다.
- **자기지도학습:** 이미지의 일부를 가리고 복원하는 과정에서 모델이 이미지의 특징을 파악하는 법을 학습한다.
- **이미지-텍스트 공동 학습:** 사진과 그 사진을 설명하는 문장을 함께 보여주어 이미지와 문장이 잘 맞는지 판단하는 표현을 학습한다.

**언어 분야**

- 대규모 말뭉치로 모델을 사전 학습하고, 작업에 맞게 추가 학습하거나 활용한다.
- **BERT로 문장을 분류하는 경우:** 문장의 특징을 파악하여 긍정·부정 또는 정상·이상 등을 구분하도록 Fine-tuning할 수 있다.
- **GPT 같은 생성형 LLM으로 답변을 만드는 경우:** 기존에 학습한 문장 생성 구조를 활용하고, 질문과 답변 예시를 추가 학습하거나 기존 언어 능력을 바탕으로 답변을 생성한다.

### ③ Foundation Model

**Foundation Model**은 여러 작업에 활용할 수 있도록 방대한 데이터로 사전 학습한 기반 모델이다.

- 문맥과 모양을 파악하거나 문장을 생성하는 능력을 갖추고 있다.
- 하나의 모델을 여러 작업의 출발점으로 활용할 수 있다.
- 이후 작업에 맞게 추가 학습하거나 프롬프트를 사용하여 활용할 수 있다.

> Foundation Model은 여러 작업의 기반이 되는 사전 학습 모델을 가리킨다.

### ④ Fine-tuning 정의 및 과정

**Fine-tuning**은 사전 학습된 모델을 새 작업의 데이터로 추가 학습하고, 기존 모델의 가중치 일부 또는 전체를 조정하는 과정이다.

1. **특징 추출:** 기존 모델이 이미지에서 다양한 특징을 추출한다.
2. **예측:** 새로운 분류층을 통해 정상·폐렴 여부를 예측한다.
3. **오차 계산:** 모델의 예측 결과와 실제 정답을 비교하여 손실(Loss)을 계산한다.
4. **가중치 업데이트:** 역전파(Backpropagation)를 통해 손실을 줄이는 방향으로 기존 가중치를 구하고, 학습 대상에 맞게 설정된 가중치를 업데이트한다.

- 모델의 모든 가중치를 반드시 업데이트할 필요는 없다.
- 어떤 가중치를 학습 대상으로 설정하느냐에 따라 새로운 작업에 적용할 수 있는 범위와 학습에 필요한 비용이 달라진다.

### ⑤ Fine-tuning 방법

#### Full Fine-tuning

- 모델 전체를 학습하는 방식이다.
- 사전 학습된 모델의 모든 가중치를 업데이트한다.
- 새로운 작업의 특징을 폭넓게 학습할 수 있다.
- 전체 가중치를 업데이트하므로 학습에 필요한 메모리와 연산 비용이 크다.

#### Partial Fine-tuning

- 재사용할 층은 고정하고 필요한 층만 학습하는 방식이다.
- 앞쪽 층은 선·윤곽·밝기 차이와 같이 다양한 이미지에 공통으로 활용할 수 있는 비교적 단순한 특징을 학습한다.
- 뒤쪽 층은 앞쪽에서 추출한 특징을 조합하여 특정 작업에 필요한 복잡한 특징을 학습한다.
- 따라서 새 작업에 필요한 층을 선택해 학습하고 나머지 층은 고정할 수 있다.

#### Frozen Backbone

- 사전 학습된 Backbone의 가중치는 고정하고, 새롭게 추가한 분류층만 학습하는 방식이다.
- 기존 모델의 특징을 그대로 사용하므로 학습할 파라미터가 적다.
- 기존 모델이 새 작업에 필요한 특징을 충분히 학습하지 못했다면 성능이 제한될 수 있다.

| 구분 | 데이터 적용 | 데이터 닮음 | 가중치 업데이트 |
|---|---|---|---|
| Frozen Backbone | 기존 모델과 유사 | 유사 | 새 분류층 중심 |
| Partial | 기존 모델과 유사 | 유사 | 일부 층 |
| Full | 기존 모델과 다름 | 다름 | 전체 가중치 |

## Fine-tuning의 한계를 넘어서는 방법

### ① PEFT(Parameter-Efficient Fine-Tuning)

**PEFT**는 사전 학습 모델의 가중치 대부분 또는 전부를 고정하고, 새 작업에 필요한 최소한의 파라미터만 학습하는 방법이다.

- 학습 중 업데이트할 파라미터가 줄어들기 때문에 메모리 부담이 낮아진다.
- 학습이 끝난 뒤에도 전체 모델이 아닌 변경된 파라미터를 따로 저장할 수 있다.
- 여러 작업에 맞는 모델을 각각 만들 때 저장 공간과 자원 부담을 줄이는 데 활용할 수 있다.

### ② PEFT 구현 방법

#### LoRA(Low-Rank Adaptation)

- 기존 모델에 더할 수정량을 작은 두 행렬로 학습하는 방식이다.
- 기존 가중치 `W`에 변화량 `ΔW`를 더한다.

$$
W_{new} = W + \Delta W = W + BA
$$

- `W`: 고정된 기존 가중치
- `ΔW`: 기존 가중치에 더할 변화량
- `A`, `B`: 작은 행렬 두 개

- 전체 가중치 행렬을 직접 업데이트하는 대신 작은 행렬 A와 B를 학습하므로 학습 파라미터 수를 줄일 수 있다.
- LoRA의 rank를 작게 설정하면 학습 파라미터 수가 줄어들고 메모리와 연산 비용도 감소할 수 있다.

#### Adapter

- 기존 모델 중간에 작은 신경망을 넣는 방식이다.
- 기존 모델의 가중치는 대부분 고정하고, 추가한 Adapter의 가중치만 학습한다.

#### Prompt Tuning

- 모델 입력에 학습 가능한 작은 벡터를 넣는 방식이다.
- 입력을 작은 숫자 벡터로 변환하고, 기존 입력에 학습용 벡터를 더해 학습한다.
- 모델 본체의 가중치는 고정한 채 입력 표현을 조정할 수 있다.

### ③ Zero-shot & Few-shot

#### Few-shot

- 모델에게 예시를 프롬프트에 제공하는 방식이다.
- 모델은 예시를 참고해 규칙과 형식을 파악하고 작업을 수행한다.

#### Zero-shot

- 원하는 작업을 예시 없이 지시하는 방식이다.
- 모델은 사전 학습 시 익힌 지식을 바탕으로 작업을 수행한다.

> Zero-shot은 지시만 제공하고, Few-shot은 지시와 예시를 함께 제공한다.

## 전이학습의 한계

### ① Negative Transfer

- 전이학습을 적용한 뒤 오히려 새 작업에서의 성능이 낮아지는 경우다.
- 기존에 학습한 특징이 새 작업에 유용하지 않거나, 서로 다른 특징이 충돌할 때 발생할 수 있다.

### ② Catastrophic Forgetting

- 새 데이터를 학습하면서 기존에 학습했던 지식을 잃어버리는 경우다.
- 기존 작업에서 잘하던 모델이 새 작업의 Fine-tuning 이후 기존 작업에서 성능이 떨어질 수 있다.
- 강의자료에서는 이를 완화하기 위한 방법으로 EWC(Elastic Weight Consolidation)를 제시한다.

### ③ Bias Transfer

- 사전 학습에서 얻은 편향이 새 작업에도 이어지는 경우다.
- 사전 학습에서 배운 특징이나 패턴에 편향이 포함되어 있다면, 새 작업의 데이터에도 그 편향이 반영될 수 있다.

### ④ Transferability 측정의 어려움

- 사전 학습 모델이 배운 지식이 새 작업에 얼마나 잘 활용될 수 있는지를 측정하는 것은 어렵다.
- 같은 흉부 X-ray를 분류하더라도 모델마다 잘 알고 있는 지식이 다르기 때문에, 어떤 사전 학습 모델이 더 유용한지 판단하기 어렵다.
- 강의자료에서 관련 지표로 LEEP, LogME를 제시한다.

## 전이학습 사용 사례

### ① 분야별 사례

**CV - 의료 영상 / X-ray에서 질환 분류**

- 일반 이미지로 사전 학습한 모델을 활용한다.
- 흉부 X-ray로 Fine-tuning한다.
- 정상과 폐렴을 분류한다.

**NLP - 법률 문서 / 계약서 조항 분류**

- 대규모 텍스트로 사전 학습한다.
- 특정 계약서 조항을 분류하는 데이터로 Fine-tuning한다.
- 계약서의 조항 유형을 분류한다.

## 딥러닝 모델 경량화

### ① 경량화 4가지 방법

#### 가지치기(Pruning)

**모델에서 예측에 미치는 영향이 작다고 판단한 부분을 제거하는 방법**이다.

- 중요도가 낮은 연결을 제거하거나, 뉴런·블록과 같은 구조를 제거한다.
- 연결만 제거하는 비구조적 가지치기와 뉴런·블록 등을 제거하는 구조적 가지치기가 있다.
- 필요에 따라 가지치기 이후 추가 학습을 수행하여 성능 저하를 보완할 수 있다.

#### 양자화(Quantization)

**모델이 쓰는 숫자를 더 적은 비트로 표현하여 저장 공간을 줄이는 방법**이다.

- 기존 모델은 FP32와 같은 정밀도로 가중치를 저장할 수 있다.
- 이를 FP16이나 INT8 등 더 낮은 정밀도로 변환한다.
- 저장 공간과 메모리 사용량을 줄일 수 있다.
- 정밀도가 낮아지므로 성능이 저하될 수 있다.

#### 지식 증류(Knowledge Distillation)

**큰 모델(Teacher Model)의 지식을 작은 모델(Student Model)에 전달하는 방법**이다.

- Teacher Model은 정답 라벨뿐 아니라 예측 확률분포도 제공한다.
- Student Model은 Teacher Model의 출력 분포를 함께 학습한다.
- Student Model이 Teacher Model의 모든 능력을 그대로 갖추는 데에는 한계가 있지만, 더 적은 메모리와 계산량으로 활용할 수 있다.

#### 모델 구조 경량화

**모델을 설계할 때부터 같은 작업을 더 적은 계산으로 수행할 수 있는 구조를 선택하는 방법**이다.

- 모델을 구성할 때 불필요한 연산을 줄이는 구조를 선택한다.
- 예를 들어 고양이와 강아지를 구분하는 문제에서는 필요한 특징을 추출하는 데 적합한 모델 구조를 선택할 수 있다.

### ② 경량화 방법 선택 기준

| 선택 기준 | 상황 | 고려할 방법 |
|---|---|---|
| 재학습 가능 여부 | 재학습이 어려움 | 양자화 |
| 재학습 가능 여부 | 데이터와 재학습 자원이 있음 | 가지치기 후 추가 학습, 지식 증류, 모델 구조 경량화 |
| 메모리·속도 | 메모리가 부족함 | 양자화 |
| 메모리·속도 | 실행 속도가 느림 | 구조적 가지치기, 모델 구조 경량화 |
| 허용할 성능 손실 | 거의 허용하지 않음 | 더 낮은 비트 양자화 또는 가지치기 정도를 신중히 선택 |

---

# AI Alignment

## AI Alignment란?

### ① 기존 한계점

- LLM은 사전학습에서 앞에 나온 토큰을 보고 다음 토큰을 예측하는 방법을 배운다.
- 이 과정에서 다음 토큰을 잘 예측하는 능력과 사람이 원하는 방향으로 답하는 능력이 반드시 일치하지는 않는다.
- 따라서 모델이 사용자의 의도와 다르게 답하거나 사실과 다른 내용을 생성할 수 있다.

### ② AI Alignment 정의

**AI Alignment(인공지능 정렬)**는 모델이 사람의 의도와 목표를 이해하고, 사람이 중요하게 여기는 가치에 맞게 행동하도록 만드는 과정이다.

- 모델이 실제로 무엇을 목표로 삼고 있는지 이해해야 한다.
- 모델이 학습을 통해 어떤 목표나 전략을 익혔는지도 확인해야 한다.

### ③ 정렬 기준

강의자료에서는 정렬 기준으로 다음 세 가지를 제시한다.

- **Helpful:** 사용자의 의도를 파악하고 실제로 도움이 되는 답변을 제공한다.
- **Honest:** 사실을 바탕으로 정직하게 답변한다.
- **Harmless:** 사람에게 해를 끼치지 않도록 한다.

## AI Alignment의 두 가지 관점

### ① Outer Alignment

**Outer Alignment는 목표를 제대로 설정했는가?**에 관한 관점이다.

- 사람이 원하는 목표를 보상 기준이나 목표 함수에 제대로 반영했는지 살펴본다.
- 보상 기준이 허점이 있으면 모델이 실제 목표와 다른 방식으로 높은 점수를 얻을 수 있다.

#### 보상 해킹(Reward Hacking)

- 보상 기준의 허점을 이용해 점수만 높이는 현상이다.
- 예를 들어 방 청소를 잘했는지 카메라로 평가한다고 가정한다.
- 물건을 정리해도 점수를 주고, 카메라를 가려 물건이 보이지 않게 해도 점수를 준다면 모델은 실제로 방을 정리하지 않고 카메라를 가릴 수 있다.
- 점수는 높아졌지만, 방을 정리한다는 원래 목표는 달성하지 못한 것이다.

### ② Inner Alignment

**Inner Alignment는 모델이 실제로 무엇을 배웠는가?**에 관한 관점이다.

- 사람이 적절한 목표를 설정했더라도 모델이 학습 과정에서 의도와 다른 목표나 전략을 익힐 수 있다.
- 모델의 내부에서 어떤 목표와 전략이 형성되었는지 살펴봐야 한다.

#### Deceptive Alignment(기만적 정렬)

- 모델이 다른 목표를 숨긴 채 평가를 통과하기 위한 수단으로 훈련이나 감독 중에는 사람의 의도에 맞게 행동할 수 있다는 관점이다.
- 모델이 실제로 어떤 목표를 따르는지 겉으로 드러나는 행동만으로 판단하기 어렵다.

#### Interpretability(해석 가능성)

- 모델 내부의 의도와 판단 과정을 이해하는 데 필요한 접근이다.
- 모델이 어떤 기준으로 결과를 선택했는지 파악할 수 있도록 내부 동작을 살펴보는 관점이다.

## 정렬을 어렵게 만드는 요인

### ① 가치 정렬의 모호성

- 사람마다 가치관이 달라 모두가 동의하는 기준을 만들기 어렵다.
- 같은 질문에 대해서도 사람마다 바람직하다고 생각하는 답변이 다를 수 있다.

### ② 확장 가능한 감독의 한계

- 사람이 직접 확인할 수 있는 범위에는 한계가 있다.
- 모델이 더 복잡한 일을 수행할수록 모든 행동과 결과를 사람이 직접 확인하기 어렵다.

## RLHF

### ① RLHF(Reinforcement Learning from Human Feedback) 정의

**RLHF**는 사람의 피드백을 통해 모델이 원하는 방향으로 답변하도록 학습하는 방법이다.

- 사람이 선호하는 답변을 모델이 학습하도록 한다.
- 사람이 직접 정답 문장을 모두 작성하는 대신, 여러 답변 중 어떤 답변을 더 선호하는지 비교하는 피드백을 활용한다.

### ② InstructGPT의 고전적 RLHF 파이프라인: SFT → Reward Model → RL

#### 1. SFT(Supervised Fine-Tuning)

- 사람이 작성한 질문과 모범 답변 데이터를 사용한다.
- 모델이 질문에 맞는 답변을 생성하도록 지도 학습한다.
- 답변을 생성하는 기본 모델을 만든다.

#### 2. Reward Model(보상 모델)

- 사람이 여러 답변을 비교하고 선호하는 답변을 선택한다.
- 선택 결과를 학습하여 답변의 선호도를 점수로 변환하는 모델을 만든다.
- 사람이 선호하는 답변에는 더 높은 점수를, 선호하지 않는 답변에는 더 낮은 점수를 주도록 학습한다.

#### 3. RL(Reinforcement Learning, 강화학습)

- SFT를 마친 언어 모델이 답변을 생성한다.
- Reward Model이 답변에 대한 보상 점수를 계산한다.
- 모델은 보상 점수를 높이는 방향으로 업데이트된다.
- 강의자료에서는 PPO(Proximal Policy Optimization)를 강화학습 알고리즘의 예로 제시한다.

> **RLHF의 흐름:** SFT로 답변 생성 모델을 만들고, Reward Model로 사람의 선호도를 점수화한 뒤, RL을 통해 보상을 높이는 방향으로 모델을 조정한다.

### ③ 선호 쌍의 보상 차이

- 같은 질문에 대한 두 답변을 비교하여 선호하는 답변과 선호하지 않는 답변을 구분한다.
- `r_chosen`은 사람이 선호한 답변의 보상 점수이고, `r_rejected`는 선호하지 않은 답변의 보상 점수다.
- 선호한 답변의 점수가 더 높아지도록 두 점수의 차이를 학습한다.

### ④ 부작용: Reward Hacking / Sycophancy

#### Reward Hacking(보상 해킹)

- 보상 점수를 높이는 방향으로 학습하다가, 실제로 좋은 결과를 만드는 대신 보상 점수만 높이는 행동을 학습할 수 있다.
- 내용의 정확성보다 높은 보상을 받도록 특정 말투나 형식을 과도하게 사용할 수도 있다.

#### Sycophancy(아첨)

- 모델이 사실이나 근거보다 사용자의 생각에 무조건 동의하는 현상이다.
- 사용자의 의견에 맞춰 답변하면서 정직성과 일관성이 떨어질 수 있다.

---

# 프롬프트 엔지니어링

## 프롬프트 엔지니어링이란?

### ① 프롬프트 엔지니어링의 개념

**프롬프트 엔지니어링(Prompt Engineering)**은 모델을 다시 학습시키지 않고, 입력 프롬프트를 설계하여 원하는 결과를 얻는 방법이다.

- 질문을 모호하게 작성하면 모델은 조건을 정확히 알지 못한 채 답변할 수 있다.
- 지역, 예산, 상황, 개수, 형식 등을 구체적으로 정하면 원하는 답변을 얻기 쉽다.
- 모델의 파라미터를 바꾸는 것이 아니라 입력을 설계한다.

**식당에서 메뉴를 주문하는 예시**

- 요리사(모델)에게 "맛있는 거 해주세요"라고만 말하면 원하는 메뉴가 무엇인지 알기 어렵다.
- 반면 지역, 예산, 상황, 인원, 형식 등을 구체적으로 전달하면 원하는 결과에 가까운 답을 얻을 수 있다.

### ② Prompt vs Fine-tuning, 언제 무엇을 쓸까?

| 구분 | 프롬프트 엔지니어링 | Fine-tuning |
|---|---|---|
| 비용 | 낮음(API 호출 비용 정도) | 높음(데이터 준비, GPU 학습) |
| 속도 | 빠름. 시도와 수정이 즉시 가능 | 학습 시간이 필요함 |
| 적용 방식 | 모델이 입력된 조건을 바탕으로 답변 | 모델 자체를 특정 작업에 맞게 학습 |
| 사용 상황 | 같은 모델이 조건에 따라 다른 답을 해야 할 때 | 모델이 특정 작업에 반복적으로 맞춰야 할 때 |

> 프롬프트는 비용이 적게 들고 바로 수정할 수 있어 보통 프롬프트부터 시도한다. 그래도 원하는 결과가 나오지 않는다면 Fine-tuning을 고려한다.

## 주요 기법

### ① Zero-shot / Few-shot

모델에게 입력을 어떻게 수행할지 알려주는 방식이다.

- **Zero-shot:** 예시 없이 지시만 주는 방식
- **Few-shot:** 입력과 출력의 예시를 몇 개 보여주어 같은 형식으로 답하게 하는 방식

**예시: 영화 리뷰 감정 분류**

- 질문만 주면 감정 분류 기준이 명확하지 않아 답변 형식이나 긍정·부정 판단이 달라질 수 있다.
- Zero-shot에서는 긍정·부정·중립 등 분류 기준을 지시한다.
- Few-shot에서는 각 감정에 해당하는 예시와 결과를 함께 제공한다.

### ② Chain-of-Thought(CoT)

- 복잡한 문제를 해결할 때 중간 단계를 거쳐 문제를 푸는 방식이다.
- 단답을 요구하는 대신 단계별 풀이를 유도한다.
- 문제를 작은 단계로 나누어 해결하고, 각 단계의 결과를 확인하도록 한다.

**강의자료 예시: 손흥민이 월드시리즈에서 선발 투수로 나섰는가?**

- 단답만 요구하면 문장 속 인물과 대회의 의미를 확인하지 않고 답할 수 있다.
- 단계별 풀이를 유도하면 먼저 손흥민이 축구 선수인지 확인하고, 월드시리즈가 미국 프로야구(MLB)의 우승팀을 가리는 경기인지 확인한다.
- 마지막으로 축구 선수가 야구 경기에서 투수로 나서는 것이 말이 되는지 판단하여 답한다.
- 강의자료의 예시 답변은 **아니오**이다.

### ③ Role Prompting

- 모델에게 역할을 부여하여 답변의 관점이나 말투, 설명 수준을 조정하는 방식이다.
- 같은 질문도 역할에 따라 관점과 강조하는 부분이 달라질 수 있다.

**예시**

- 역할 없음: "금리 인상이 뭐야?"
- 은행 PB: "너는 10년 차 은행 PB야. 금리 인상이 뭐야?"
- 한국은행 경제학자: "너는 한국은행 경제학자야. 금리 인상이 뭐야?"

> 역할을 정해주면 설명하는 관점이나 말투는 달라질 수 있지만, 모델이 아는 내용의 범위 자체가 늘어나는 것은 아니다.

### ④ Instruction Prompting

- 원하는 것을 구체적으로 적어서 지시하는 방식이다.
- 답의 형식이나 길이, 누구를 위한 답인지, 꼭 들어가야 할 내용 등을 명확하게 제시한다.

**예시**

- 요청만 한 경우: "이 기사 요약해줘."
- 형식과 제약을 지정한 경우: "이 기사를 3문장 이내로, 중학생이 이해할 수 있는 말로 요약해줘. 핵심 수치와 결과를 포함해줘."

### ⑤ Self-Consistency

- 같은 문제를 여러 번 풀게 하고, 여러 답변 중 가장 많이 나온 답을 선택하는 방식이다.
- 한 번의 풀이 결과에만 의존하지 않고 여러 번의 결과를 비교한다.
- 여러 번 같은 답이 나오는지 확인하여 답을 선택한다.

### ⑥ 기법별 사용 상황

| 기법 | 핵심 아이디어 | 적합한 상황 |
|---|---|---|
| Zero-shot | 예시 없이 지시만 줌 | 간단하고 명확한 작업 |
| Few-shot | 예시 몇 개 보여줌 | 출력 형식이나 기준을 알려주고 싶을 때 |
| Chain-of-Thought | 풀이 과정을 단계별로 나눔 | 계산, 논리, 순서가 필요한 문제 |
| Role Prompting | 역할을 부여함 | 관점·말투·설명 수준을 조정할 때 |
| Instruction Prompting | 지시 내용을 구체화함 | 결과 형식과 제약 조건을 분명히 할 때 |
| Self-Consistency | 여러 번 풀고 다수결로 선택 | 답변의 일관성을 확인하고 싶을 때 |

> 일반적으로는 원하는 것을 명확하게 지시하는 Instruction Prompting부터 시작한다. 필요에 따라 Few-shot, CoT, Self-Consistency를 조합한다.

## 프롬프트 엔지니어링의 한계

### ① Prompt Injection(프롬프트 주입)

- 사용자가 입력한 내용에 모델의 기존 지시를 무시하도록 유도하는 문장이 포함되는 문제다.
- 모델이 사용자 입력을 시스템의 지시보다 우선하여 따를 수 있다.

**예시**

1. 개발자가 시스템 프롬프트에 "사용자가 입력한 문장을 영어로 번역해줘"라고 지시한다.
2. 사용자가 "오늘 날씨 좋다"라는 문장과 함께 다른 지시를 입력한다.
3. 모델이 번역 대신 사용자 입력에 포함된 새로운 지시를 따를 수 있다.

**완화 방법**

- 시스템 지시와 사용자 입력 사이에 구분자를 넣어 경계를 알려준다.
- 모델이 수행할 수 있는 권한을 최소화한다.
- 모델의 출력을 그대로 사용하지 않고 한 번 더 검증한다.

### ② Hallucination(환각)

- 모델이 실제로 없는 내용을 그럴듯하게 만들어내는 문제다.
- 존재하지 않는 논문이나 정보를 실제 자료처럼 답할 수 있다.

**완화 방법**

- 답변의 근거가 될 자료를 함께 제공한다(RAG).
- 출처를 요구하고 직접 확인한다.
- 모델의 답변을 자료와 대조하여 검증한다.

### ③ 모델 의존성

- 같은 프롬프트라도 모델이나 버전이 바뀌면 결과가 달라질 수 있다.
- 같은 질문에 긍정·부정 표현이나 답변 형식이 달라지는 문제가 발생할 수 있다.

**완화 방법**

- 모델 버전을 고정하여 사용한다.
- 자주 쓰는 질문과 기대 답변을 모아 두고, 모델이 바뀔 때마다 다시 확인하는 회귀 테스트를 수행한다.

### ④ 성능 한계

- 모델이 원래 해결하지 못하는 일을 프롬프트만으로 해결하기는 어렵다.
- 예를 들어 정확한 계산이 필요한 작업에서 모델이 틀린 계산 결과를 낼 수 있다.

**완화 방법**

- 모델이 못하는 일을 대신하는 외부 도구를 연결한다. 강의자료에서는 계산기를 예로 제시한다.
- 필요한 경우 Fine-tuning으로 모델을 해당 작업에 맞게 추가 학습한다.

### 한계 정리

| 한계 | 내용 | 완화 방법 |
|---|---|---|
| Prompt Injection | 입력에 섞인 지시가 기존 지시를 무시하게 함 | 지시와 입력 분리, 권한 최소화, 출력 검증 |
| Hallucination | 사실이 아닌 내용을 그럴듯하게 생성 | 근거 자료 제공(RAG), 출처 확인 |
| 모델 의존성 | 모델·버전에 따라 답변이 달라짐 | 모델 버전 고정, 회귀 테스트 |
| 성능 한계 | 모델이 본래 못하는 작업은 프롬프트로 해결하기 어려움 | 외부 도구 연결, Fine-tuning |

---

# 하네스 엔지니어링

## 프롬프트 다음의 질문

### ① 프롬프트만으로는 안 되는 일

- 프롬프트는 한 번의 질문과 답변으로 끝나는 일에는 유용하다.
- 하지만 실제 업무는 한 번의 질문으로 끝나지 않는 경우가 많다.

**고객센터 주문 조회 예시**

사용자가 "제 주문 어디까지 왔어요?"라고 물으면 다음 작업이 필요하다.

1. 고객의 주문 내역을 DB에서 조회한다.
2. 조회 결과가 맞는지 확인한다.
3. DB가 응답하지 않으면 다시 시도하거나 사람에게 넘긴다.

- 모델에 프롬프트를 아무리 잘 작성해도 DB에 접근할 수 없으면 주문 내역을 조회할 수 없다.
- 파일을 읽고, 도구를 쓰고, 결과를 확인하고, 실패하면 복구해야 하는 일이라면 모델 바깥에 실행 환경이 필요하다.

### ② 하네스란?(Harness)

**하네스(Harness)**는 모델을 둘러싸고 모델이 실제로 일을 할 수 있도록 필요한 도구와 정보, 규칙을 제공하는 주변 시스템이다.

- 모델 자체만으로는 외부 DB나 도구를 직접 사용할 수 없다.
- 모델이 어떤 도구를 사용할 수 있는지, 실행 결과를 어떻게 받는지, 어떤 행동이 허용되는지 등을 정해야 한다.
- 하네스에는 지시·도구·실행 환경·상태와 메모리·관측과 피드백·권한과 가드레일 등이 포함된다.

## 하네스의 구성 요소

### ① 하네스를 이루는 여섯 가지 요소

| 구성 요소 | 무엇을 정하는가? | 고객센터 에이전트 예시 |
|---|---|---|
| 지시 / 컨텍스트 | 모델이 무엇을 보고 시작하는가 | 시스템 프롬프트, 고객 정보, 이전 대화 |
| 도구 인터페이스 | 모델이 무엇을, 어떤 형식으로 호출할 수 있는가 | 주문 조회 도구, 주문 취소 도구 |
| 실행 환경 | 행동이 실제로 어디서 실행되는가 | 주문 DB에 접근하는 서버·실행 환경 |
| 상태 / 메모리 | 지금까지 뭘 했고 뭘 기억해야 하는가 | 주문 조회 결과, 이전 대화 |
| 관측 / 피드백 | 행동의 결과를 어떻게 돌려받는가 | 도구의 조회 결과, 오류 메시지 |
| 권한 / 가드레일 | 무엇은 허용되고 무엇은 막히는가 | 조회 권한, 취소 권한, 승인 조건 |

### ② 도구 스키마(Tool Schema)

- 모델에게 도구의 코드를 직접 보여주는 것이 아니라, 도구가 어떤 일을 하는지 설명하는 정보를 제공한다.
- 도구 스키마는 모델에게 **언제, 무엇을, 어떤 값으로 호출하는지** 알려주는 설명서다.
- 도구의 이름, 설명, 입력값의 형식, 필수 입력값 등을 명확하게 정의해야 한다.

**도구 스키마에 포함되는 항목**

| 항목 | 의미 |
|---|---|
| `type` | 도구의 유형 |
| `name` | 도구 이름 |
| `description` | 도구가 수행하는 작업에 대한 설명 |
| `parameters` | 도구에 전달할 입력값의 구조 |
| `properties` | 입력값의 이름과 자료형 |
| `required` | 반드시 입력해야 하는 값 |
| `enum` | 입력할 수 있는 값의 범위 |
| `additionalProperties` | 정의하지 않은 추가 입력을 허용할지 여부 |
| `strict` | 정해진 스키마를 엄격하게 따르도록 할지 여부 |

- 도구의 설명과 입력 형식이 명확하면 모델이 도구를 호출하는 행동도 달라질 수 있다.
- 예를 들어 주문 조회 도구라면 고객 ID와 주문 상태를 어떤 형식으로 전달할지 명확히 정해야 한다.

### ③ 성공과 실패의 반환 계약

- 도구 실행 결과는 성공과 실패를 구분할 수 있는 형태로 반환해야 한다.
- 실패했을 때 결과를 모두 `None`으로 반환하면 모델은 무엇이 잘못되었는지 알기 어렵다.
- 성공 여부, 오류 코드, 오류 메시지, 재시도 가능 여부 등을 구분해서 반환하면 다음 행동을 결정할 수 있다.

**주문 조회 예시**

- 주문이 정상적으로 조회되면 조회 결과를 반환한다.
- DB가 응답하지 않으면 `DB_TIMEOUT`과 같이 오류를 구분해 반환한다.
- 다시 시도할 수 있는 오류라면 `retryable: true`와 같이 재시도 가능 여부를 알려준다.
- 조회 결과가 없다는 정상적인 결과와 DB 연결 실패를 구분해야 한다.

### ④ ACI(Agent-Computer Interface)

- ACI는 에이전트와 컴퓨터가 상호작용하는 인터페이스를 뜻한다.
- 모델이 실제 도구와 환경을 어떻게 사용할지 설계하는 부분이다.
- 강의자료에서는 에이전트가 사용하는 도구와 상호작용 방식을 바꾸었을 때 작업 결과가 달라질 수 있음을 사례로 제시한다.

## 실패는 어디서 오는가?

### ① 인터페이스만 바꿔도 달라지는 결과

- 에이전트의 추론 능력만이 아니라 도구를 사용할 수 있는 방식도 결과에 영향을 준다.
- 같은 모델이라도 도구의 설명과 호출 방식이 달라지면 작업 성능이 달라질 수 있다.

### ② 증상과 수정할 곳

| 증상 | 수정할 곳(왜 여기인가?) | 관련 요소 |
|---|---|---|
| 엉뚱한 도구를 부른다 | 도구 이름, 설명, 입력 형식 등을 명확하게 작성 | 도구 인터페이스 |
| 같은 실수를 반복한다 | 도구의 반환값과 오류 메시지를 더 구체적으로 전달 | 관측 / 피드백 |
| 앞부분을 잊는다 | 모델에 필요한 정보, 진행 기록, 상태를 제공 | 지시 / 컨텍스트, 상태 / 메모리 |
| 위험한 행동을 바로 실행한다 | 허용 범위, 사람 승인, 실행 전 제한을 설정 | 권한 / 가드레일, 실행 환경 |
| 결과가 맞는지 모른다 | 테스트·검증·확인할 수 있는 수단을 마련 | 관측 / 피드백 |

> 하네스는 모델이 일을 잘하도록 돕는 것뿐 아니라, 잘못된 행동을 확인하고 제한하는 역할도 한다.

---

# 루프 엔지니어링

## 루프란 무엇인가?(Loop)

### ① 사람이 하던 반복을 시스템으로

**루프(Loop)**는 모델이 행동을 제안하고, 시스템이 그 행동을 실행한 뒤, 결과를 다시 모델에게 돌려주어 다음 행동을 결정하도록 반복하는 구조다.

- 모델이 한 번 답하고 끝나는 것이 아니라, 결과를 확인하고 다음 행동을 수행한다.
- 실제 에이전트 서비스에서는 모델이 실행할 명령을 제안하고, 실행은 도구와 실행 환경에서 이루어진다.
- 도구 실행 결과는 다시 모델에 전달되어 다음 행동을 결정하는 데 사용된다.

### ② 통제된 에이전트 루프

루프는 다음 단계로 이루어진다.

1. **관측:** 지금 상황과 이전 작업 결과를 확인한다.
2. **판단:** 다음에 무엇을 해야 하는지 결정한다.
3. **행동:** 필요한 도구를 실행한다.
4. **검증:** 결과가 목표에 맞는지 확인한다.
5. **반복 또는 종료:** 목표를 달성하지 못했다면 다시 관측하고, 달성했다면 종료한다.

### ③ 공간에서 시간으로

- **하네스:** 모델이 일하는 공간과 도구·환경을 설계한다.
- **루프:** 그 공간 안에서 어떤 순서로 행동을 반복할지 설계한다.
- 하네스가 도구와 실행 환경을 제공하면 루프는 그 도구를 실행하고 결과를 확인하며 다음 행동으로 이어간다.

## 루프의 뿌리: ReAct

### ① ReAct(Reasoning + Acting)

**ReAct**는 생각(Thought)과 행동(Action)을 번갈아 수행하면서 결과를 관찰(Observation)하는 방식이다.

- **Thought(추론):** 지금 무엇이 필요한지 판단한다.
- **Action(행동):** 검색, 도구 호출 등의 행동을 실행한다.
- **Observation(관측):** 행동의 결과를 보고 다음 생각에 반영한다.

### ② ReAct의 흐름

1. **Thought:** 주문 조회가 필요한지 판단한다.
2. **Action:** 주문 조회 도구를 호출한다.
3. **Observation:** 조회 결과 또는 오류 결과를 확인한다.
4. **Thought:** 결과를 바탕으로 다음 행동을 결정한다.
5. **Action:** 필요한 경우 도구를 다시 호출한다.
6. **Observation 및 최종 답변:** 목표 달성 여부를 확인하고 답변을 마무리한다.

- 예를 들어 주문 조회 도구가 `DB_TIMEOUT`을 반환하고 `retryable: true`라면, 다음 Thought에서 다시 시도할지 결정할 수 있다.
- 조회가 성공하면 결과를 확인하고 사용자에게 답변한다.

## 루프의 구성 요소

### ① 루프를 이루는 여섯 가지 구성 요소

| 구성 요소 | 무엇을 정하는가? | 강의자료의 예시 |
|---|---|---|
| 목표 | 무엇을 끝내려 하는가 | 레거시 API를 사용하는 파일을 새 API로 전환 |
| 트리거 | 루프가 언제 시작되는가 | 사람이 요청할 때, 매일 정해진 시간, 코드가 올라올 때 |
| 관측과 행동 | 매 반복에서 무엇을 보고 무엇을 하는가 | 파일 검색, 코드 수정, 테스트 실행 |
| 정상 종료 조건 | 성공해서 멈추는 조건 | 레거시 API 사용 0건 및 전체 테스트 통과 |
| 안전 상한 | 위험해서 멈추는 조건 | 반복 200회, 60분, 20달러 등 상한 설정 |
| 실패 / 승인 처리 | 막혔을 때 어떻게 하는가 | 한 번 재시도한 뒤 멈추고 보고, 삭제·배포는 사람 승인 |

### ② 정상 종료 조건과 위험해서 멈추는 조건

- **성공해서 멈추는 조건:** 목표를 달성했거나 정상 종료 조건을 통과했을 때 종료한다.
- **위험해서 멈추는 조건:** 반복·시간·비용 상한에 도달하거나, 위험한 행동이라 사람의 승인이 필요한 경우 실행을 멈춘다.
- 정상 종료 조건과 안전 상한은 함께 설정해야 한다.

### ③ 루프 패턴과 사용 상황

| 패턴 | 사용 상황 | 종료 조건 | 안전 상한 |
|---|---|---|---|
| 재시도 | 일시적 오류(예: `DB_TIMEOUT`) | 성공하면 종료 | 최대 재시도 횟수, 재시도 간격 |
| 주기 실행 | 매일 아침 리포트 생성 | 해당 회차 작업 완료 | 회차당 시간·비용 상한, 실패 시 알림 |
| 이벤트 트리거 | 코드가 올라오면 자동 리뷰 | 리뷰 작성 완료 | 같은 이벤트 중복 실행 방지, 회당 시간 상한 |
| 목표 달성까지 반복 | 레거시 API 마이그레이션 | 레거시 API 사용 0건 및 전체 테스트 통과 | 반복 횟수·시간·비용 상한 |

## 루프의 통제 장치와 실행 단계

### ① 최소 루프의 Python 의사코드

강의자료에서는 루프의 구성 요소를 다음과 같은 최소 코드로 표현한다.

```python
def run_agent_loop(agent, state, max_steps=20):
    for step in range(max_steps):
        decision = agent.next(state)

        if decision.kind == "final":
            return {"status": "completed", "output": decision.output}

        if decision.risk == "high":
            return {"status": "needs_approval", "action": decision}

        result = run_tool(decision)
        state.append({"action": decision, "result": result})

        if goal_reached(state):
            return {"status": "completed", "output": state}

    return {"status": "stopped", "reason": "max_steps"}
```

- `agent.next()`: 모델이 다음 행동을 정한다.
- `run_tool()`: 도구를 실행한다.
- `state.append()`: 결과를 기록해 다음 관측에 반영한다.
- `goal_reached()`: 목표에 도달했는지 확인한다.
- `max_steps`: 최대 반복 횟수를 제한한다.
- 위험도가 높은 행동은 바로 실행하지 않고 사람 승인을 기다린다.

### ② 무한 반복과 반복 실패

루프에서 발생할 수 있는 실패는 다음과 같다.

| 실패 | 어떻게 나타나는가? |
|---|---|
| 무한 반복 | 종료 조건이 없어 같은 시도를 계속 반복한다. |
| 토큰 / 비용 폭주 | 반복 호출이 쌓이면서 비용이 증가한다. |
| 컨텍스트 누적 | 기록이 쌓여 중요한 정보와 불필요한 정보가 섞인다. |
| 같은 부작용의 반복 | 상태 변화 없이 같은 실패 행동을 반복한다. |
| 되돌리기 어려운 행동 | 삭제·배포·승인 등 실행 후 되돌리기 어려운 작업이 자동으로 수행된다. |

### ③ 통제 장치

| 통제 장치 | 막는 실패 |
|---|---|
| 반복 횟수 제한 | 무한 반복 방지 |
| 시간·비용 상한 | 토큰과 실행 비용의 과도한 증가 방지 |
| 테스트 완료 조건 | 작업이 끝나지 않았는데 완료로 처리되는 상황 방지 |
| 중복 실행 방지 | 같은 요청이나 행동이 반복 실행되는 상황 방지 |
| 체크포인트와 복구 | 실행 전 상태를 기록하고 문제가 생겼을 때 되돌릴 수 있도록 준비 |
| 위험도에 따른 행동 분리 | 되돌리기 어려운 행동을 자동 실행하지 않도록 제한 |

### ④ 위험도에 따른 행동 분리

위험도가 높은 행동은 다음과 같이 단계를 나누어 처리한다.

1. **자동 실행:** 위험도가 낮고 허용된 행동을 실행한다.
2. **사후 검토:** 실행 결과를 확인하고 문제가 없는지 검토한다.
3. **사람 승인 후 실행:** 위험도가 높거나 되돌리기 어려운 행동은 사람의 승인을 받은 뒤 실행한다.

- 조회나 테스트 실행은 바로 실행할 수 있다.
- 되돌릴 수 있는 파일 수정이나 브랜치 커밋은 실행 후 검토할 수 있다.
- 삭제, 배포, 결제처럼 되돌리기 어렵거나 외부에 영향을 주는 행동은 사람의 승인을 받은 뒤 실행한다.
- 실제 서비스에서는 도구마다 읽기·쓰기 권한, 되돌릴 수 있는지, 계정 권한과 금전적 영향 등을 고려해 위험도를 나눈다.

### ⑤ 학습 단계의 정렬과 실행 단계의 통제

- RLHF는 SFT → Reward Model → RL 과정을 통해 모델이 사람이 선호하는 방향으로 답변하도록 학습한다.
- 그러나 학습만으로 실제 환경에서 발생할 수 있는 모든 위험을 막을 수는 없다.
- 따라서 실행 단계에서는 권한 최소화, 사람 승인, 결과 검증, 반복 횟수 제한 등의 통제 장치를 함께 사용해야 한다.

> 모델이 바람직한 답변을 하도록 학습시키는 것과, 실제 환경에서 위험한 행동을 하지 못하도록 실행을 통제하는 것은 서로 다른 단계다.